## ROI-first star centroiding on fish crops

This notebook resets the workflow around a fish ROI instead of searching the whole frame.

1. Pick one image.
2. Define a fish ROI manually on the resized image.
3. Expand that ROI and crop into the fish region.
4. Build a laser-likelihood map only inside the crop.
5. Run **FGF / star centroiding** on the brightest crop-local patch.
6. Map the centroid back to full-image coordinates.

This notebook intentionally starts with a **manual fish ROI** so we can validate the ROI-first idea before adding a fish detector model.

In [ ]:
import sys
from pathlib import Path

import cv2
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
from PIL import Image

ROOT = Path.cwd().resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

DATA_DIR = ROOT / "e4e-data"
MAX_SIDE = 1600
ROI_PAD_FRAC = 0.15
PATCH_PAD = 18
LASER_COLOR = "auto"  # "auto", "red", or "green"

# Change this file name to work on a different image.
SELECTED_FILE = "00a1a8e5c7d5f70fc791667eef155bbb.JPG"

# Manual fish ROIs on the resized image, as (x0, y0, x1, y1).
# Edit or add entries here as you inspect more frames.
ROI_PRESETS = {
    "00a1a8e5c7d5f70fc791667eef155bbb.JPG": (430, 250, 620, 430),
}

paths = sorted(DATA_DIR.glob("*.[jJ][pP][gG]")) + sorted(DATA_DIR.glob("*.[jJ][pP][eE][gG]"))
selected_path = next((p for p in paths if p.name == SELECTED_FILE), None)

print(f"ROOT={ROOT}")
print(f"Found {len(paths)} images in {DATA_DIR} (MAX_SIDE={MAX_SIDE})")
for p in paths:
    print(" ", p.name)

if selected_path is None and paths:
    selected_path = paths[0]
    print(f"\nSELECTED_FILE was not found, falling back to {selected_path.name}")
elif selected_path is not None:
    print(f"\nSelected image: {selected_path.name}")


### 1) Synthetic sanity check

Before touching real fish crops, confirm that `fgf_full` still recovers the center of a simple Gaussian spot.

In [ ]:
from fgf_full import fgf_full


def make_gaussian_patch(h=64, w=64, xc_true=31.7, yc_true=28.2, sx=2.1, sy=1.9, A=0.95, noise=0.02, seed=0):
    rng = np.random.default_rng(seed)
    y, x = np.mgrid[0:h, 0:w].astype(np.float64)
    g = A * np.exp(-((x - xc_true) ** 2) / (2 * sx**2) - ((y - yc_true) ** 2) / (2 * sy**2))
    g = np.clip(g + rng.normal(0, noise, g.shape), 0, 1)
    return g.astype(np.float32), float(xc_true), float(yc_true)


syn, tx, ty = make_gaussian_patch()
xc, yc, sx, sy, A, conf = fgf_full(syn.astype(np.float64))
err = float(np.hypot(xc - tx, yc - ty))

print(f"True center: ({tx:.3f}, {ty:.3f})")
print(f"FGF center:  ({xc:.3f}, {yc:.3f})  sigma=({sx:.3f}, {sy:.3f})  conf={conf:.3f}")
print(f"Pixel error: {err:.4f}")

fig, ax = plt.subplots(figsize=(5, 4))
ax.imshow(syn, cmap="magma", origin="upper")
ax.plot(tx, ty, "c+", ms=14, mew=2, label="true")
ax.plot(xc, yc, "yo", ms=8, mfc="none", mew=2, label="FGF")
ax.legend(loc="lower right", fontsize=9)
ax.set_title("Synthetic Gaussian + FGF centroid")
ax.axis("off")
plt.tight_layout()
plt.show()


### 2) Pick one image and define the fish ROI

The ROI is a manual fish box on the **resized image** in `(x0, y0, x1, y1)` form. Edit `ROI_PRESETS` above until the rectangle cleanly covers the fish body, then rerun the next cell.

In [ ]:
def load_image(path):
    bgr = cv2.imread(str(path), cv2.IMREAD_COLOR)
    if bgr is not None:
        return bgr

    with Image.open(path) as img:
        rgb = img.convert("RGB")
    return cv2.cvtColor(np.array(rgb), cv2.COLOR_RGB2BGR)


def resize_long_edge(bgr, max_side):
    if max_side is None:
        return bgr
    h, w = bgr.shape[:2]
    longest = max(h, w)
    if longest <= max_side:
        return bgr
    scale = max_side / longest
    return cv2.resize(
        bgr,
        (int(round(w * scale)), int(round(h * scale))),
        interpolation=cv2.INTER_AREA,
    )


if selected_path is None:
    print("No images were found in e4e-data/.")
else:
    selected_bgr = resize_long_edge(load_image(selected_path), MAX_SIDE)
    selected_rgb = cv2.cvtColor(selected_bgr, cv2.COLOR_BGR2RGB)
    FISH_ROI = ROI_PRESETS.get(selected_path.name)

    print(f"Selected image shape: {selected_bgr.shape[1]} x {selected_bgr.shape[0]}")
    print(f"Fish ROI preset: {FISH_ROI}")

    fig, ax = plt.subplots(figsize=(10, 7))
    ax.imshow(selected_rgb)
    ax.set_title(selected_path.name)
    if FISH_ROI is not None:
        x0, y0, x1, y1 = FISH_ROI
        rect = Rectangle((x0, y0), x1 - x0, y1 - y0, fill=False, ec="yellow", lw=2)
        ax.add_patch(rect)
        ax.text(x0, max(0, y0 - 10), "fish ROI", color="yellow", fontsize=10, weight="bold")
    else:
        ax.text(20, 30, "Add this image to ROI_PRESETS", color="yellow", fontsize=11, weight="bold")
    ax.axis("off")
    plt.tight_layout()
    plt.show()


### 3) ROI-first laser localization helpers

These helpers keep the real-image workflow inside this notebook: expand the fish box, crop it, score red or green laser-like pixels inside the crop, refine the best peak with FGF, and translate the centroid back to the full frame.

In [ ]:
def clamp_roi(roi, shape):
    h, w = shape[:2]
    x0, y0, x1, y1 = [int(round(v)) for v in roi]
    x0 = int(np.clip(x0, 0, w - 1))
    y0 = int(np.clip(y0, 0, h - 1))
    x1 = int(np.clip(x1, x0 + 1, w))
    y1 = int(np.clip(y1, y0 + 1, h))
    return x0, y0, x1, y1


def expand_roi(roi, shape, pad_frac=0.15):
    x0, y0, x1, y1 = clamp_roi(roi, shape)
    width = x1 - x0
    height = y1 - y0
    pad_x = int(round(width * pad_frac))
    pad_y = int(round(height * pad_frac))
    return clamp_roi((x0 - pad_x, y0 - pad_y, x1 + pad_x, y1 + pad_y), shape)


def crop_from_roi(bgr, roi, pad_frac=0.15):
    crop_roi = expand_roi(roi, bgr.shape, pad_frac=pad_frac)
    x0, y0, x1, y1 = crop_roi
    crop = bgr[y0:y1, x0:x1].copy()
    return crop, crop_roi


def robust_normalize(score, lower_q=95.0, upper_q=99.9):
    score_f = score.astype(np.float32)
    lo = float(np.percentile(score_f, lower_q))
    hi = float(np.percentile(score_f, upper_q))

    if not np.isfinite(lo):
        lo = 0.0
    if not np.isfinite(hi):
        hi = float(score_f.max())

    if hi <= lo + 1e-6:
        lo = float(score_f.min())
        hi = float(score_f.max())

    if hi <= lo + 1e-6:
        return np.clip(score_f, 0.0, 1.0).astype(np.float32)
    return np.clip((score_f - lo) / (hi - lo), 0.0, 1.0).astype(np.float32)


def detect_laser_color_crop(bgr, quantile=99.9):
    bgr_f = bgr.astype(np.float32) / 255.0
    b, g, r = cv2.split(bgr_f)
    red_score = np.clip(r - 0.5 * g - 0.5 * b, 0.0, 1.0)
    green_score = np.clip(g - 0.5 * r - 0.5 * b, 0.0, 1.0)
    red_q = float(np.percentile(red_score, quantile))
    green_q = float(np.percentile(green_score, quantile))
    return "red" if red_q >= green_q else "green"


def local_contrast_score(gray_float, kernel_size=31):
    local_mean = cv2.blur(gray_float, (kernel_size, kernel_size))
    variance = cv2.blur((gray_float - local_mean) ** 2, (kernel_size, kernel_size))
    local_std = np.sqrt(variance + 1e-8)
    contrast = (gray_float - local_mean) / (local_std + 1e-8)
    return np.clip(contrast / 5.0, 0.0, 1.0)


def compute_laser_score_crop(bgr, color="auto"):
    if color == "auto":
        color = detect_laser_color_crop(bgr)

    hsv = cv2.cvtColor(bgr, cv2.COLOR_BGR2HSV)
    lab = cv2.cvtColor(bgr, cv2.COLOR_BGR2LAB)
    bgr_f = bgr.astype(np.float32) / 255.0
    gray_f = cv2.cvtColor(bgr, cv2.COLOR_BGR2GRAY).astype(np.float32) / 255.0

    h, s, v = cv2.split(hsv)
    l, a, _ = cv2.split(lab)
    a_f = a.astype(np.float32) - 128.0

    if color == "red":
        hue_mask = ((h <= 10) | (h >= 170)).astype(np.float32)
        rgb_score = np.clip(bgr_f[:, :, 2] - 0.5 * bgr_f[:, :, 1] - 0.5 * bgr_f[:, :, 0], 0.0, 1.0)
        lab_score = np.clip(a_f / 127.0, 0.0, 1.0)
    else:
        hue_mask = ((h >= 35) & (h <= 85)).astype(np.float32)
        rgb_score = np.clip(bgr_f[:, :, 1] - 0.5 * bgr_f[:, :, 2] - 0.5 * bgr_f[:, :, 0], 0.0, 1.0)
        lab_score = np.clip(-a_f / 127.0, 0.0, 1.0)

    hsv_score = hue_mask * (s.astype(np.float32) / 255.0) * (v.astype(np.float32) / 255.0)
    brightness_score = (l.astype(np.float32) / 255.0) * lab_score
    contrast_score = local_contrast_score(gray_f)

    fused_raw = 0.40 * hsv_score + 0.25 * brightness_score + 0.20 * rgb_score + 0.15 * contrast_score
    fused = robust_normalize(fused_raw)

    debug = {
        "hsv_score": hsv_score.astype(np.float32),
        "brightness_score": brightness_score.astype(np.float32),
        "rgb_score": rgb_score.astype(np.float32),
        "contrast_score": contrast_score.astype(np.float32),
        "raw_score": fused_raw.astype(np.float32),
    }
    return fused.astype(np.float32), color, debug


def extract_patch(image, cx, cy, pad=18):
    h, w = image.shape[:2]
    cx_i = int(round(cx))
    cy_i = int(round(cy))
    x0 = max(cx_i - pad, 0)
    y0 = max(cy_i - pad, 0)
    x1 = min(cx_i + pad + 1, w)
    y1 = min(cy_i + pad + 1, h)
    return image[y0:y1, x0:x1], x0, y0


def localize_laser_in_crop(crop_bgr, laser_color="auto", patch_pad=18):
    score_map, detected_color, debug = compute_laser_score_crop(crop_bgr, color=laser_color)
    cy_peak, cx_peak = np.unravel_index(np.argmax(score_map), score_map.shape)
    peak_score = float(score_map[cy_peak, cx_peak])

    patch, patch_x0, patch_y0 = extract_patch(score_map, cx_peak, cy_peak, pad=patch_pad)
    if min(patch.shape[:2]) < 5:
        raise ValueError("FGF patch is too small; enlarge the ROI or patch size.")

    xc, yc, sx, sy, amplitude, conf = fgf_full(patch.astype(np.float64))
    local_x = float(patch_x0 + xc)
    local_y = float(patch_y0 + yc)

    return {
        "score_map": score_map,
        "laser_color": detected_color,
        "peak_score": peak_score,
        "coarse_peak": (float(cx_peak), float(cy_peak)),
        "local_xy": (local_x, local_y),
        "sigma_xy": (float(sx), float(sy)),
        "amplitude": float(amplitude),
        "fgf_confidence": float(conf),
        "patch": patch,
        "patch_origin": (int(patch_x0), int(patch_y0)),
        "debug": debug,
    }


def run_roi_first_workflow(image_path, fish_roi, pad_frac=0.15, laser_color="auto", patch_pad=18):
    image_bgr = resize_long_edge(load_image(image_path), MAX_SIDE)
    crop_bgr, crop_roi = crop_from_roi(image_bgr, fish_roi, pad_frac=pad_frac)
    localized = localize_laser_in_crop(crop_bgr, laser_color=laser_color, patch_pad=patch_pad)

    crop_x0, crop_y0, crop_x1, crop_y1 = crop_roi
    local_x, local_y = localized["local_xy"]

    localized.update({
        "image_path": Path(image_path),
        "image_bgr": image_bgr,
        "crop_bgr": crop_bgr,
        "fish_roi": clamp_roi(fish_roi, image_bgr.shape),
        "crop_roi": crop_roi,
        "full_xy": (float(crop_x0 + local_x), float(crop_y0 + local_y)),
        "crop_offset": (int(crop_x0), int(crop_y0)),
    })
    return localized


def visualize_roi_result(result):
    image_rgb = cv2.cvtColor(result["image_bgr"], cv2.COLOR_BGR2RGB)
    crop_rgb = cv2.cvtColor(result["crop_bgr"], cv2.COLOR_BGR2RGB)

    fish_x0, fish_y0, fish_x1, fish_y1 = result["fish_roi"]
    crop_x0, crop_y0, crop_x1, crop_y1 = result["crop_roi"]
    full_x, full_y = result["full_xy"]
    local_x, local_y = result["local_xy"]
    patch_x0, patch_y0 = result["patch_origin"]
    sigma_x, sigma_y = result["sigma_xy"]

    fig, axes = plt.subplots(1, 4, figsize=(20, 5))

    axes[0].imshow(image_rgb)
    axes[0].add_patch(Rectangle((fish_x0, fish_y0), fish_x1 - fish_x0, fish_y1 - fish_y0, fill=False, ec="yellow", lw=2))
    axes[0].add_patch(Rectangle((crop_x0, crop_y0), crop_x1 - crop_x0, crop_y1 - crop_y0, fill=False, ec="cyan", lw=2))
    axes[0].plot(full_x, full_y, "r+", ms=14, mew=2)
    axes[0].set_title("Full image\nfish ROI, crop ROI, final centroid")
    axes[0].axis("off")

    axes[1].imshow(crop_rgb)
    axes[1].plot(local_x, local_y, "r+", ms=14, mew=2)
    axes[1].set_title(f"Fish crop\nlaser_color={result['laser_color']}")
    axes[1].axis("off")

    im = axes[2].imshow(result["score_map"], cmap="hot")
    axes[2].plot(local_x, local_y, "c+", ms=14, mew=2)
    axes[2].set_title(f"Crop-local laser score\npeak={result['peak_score']:.3f}")
    axes[2].axis("off")
    plt.colorbar(im, ax=axes[2], fraction=0.046, pad=0.04)

    axes[3].imshow(result["patch"], cmap="magma")
    axes[3].plot(local_x - patch_x0, local_y - patch_y0, "w+", ms=14, mew=2)
    theta = np.linspace(0, 2 * np.pi, 100)
    axes[3].plot(
        (local_x - patch_x0) + sigma_x * np.cos(theta),
        (local_y - patch_y0) + sigma_y * np.sin(theta),
        "w--",
        lw=1.5,
    )
    axes[3].set_title(f"FGF patch\nconf={result['fgf_confidence']:.3f}")
    axes[3].axis("off")

    plt.tight_layout()
    plt.show()


### 4) Run the ROI-first workflow on the selected image

If the centroid lands in the wrong place, adjust the fish ROI first. The point of this notebook is to test whether a fish-first crop makes centroiding more stable before we introduce any fish model.

In [ ]:
if selected_path is None:
    print("No image is selected.")
else:
    fish_roi = ROI_PRESETS.get(selected_path.name)
    if fish_roi is None:
        print(f"Add a fish ROI for {selected_path.name} to ROI_PRESETS and rerun this cell.")
    else:
        roi_result = run_roi_first_workflow(
            selected_path,
            fish_roi,
            pad_frac=ROI_PAD_FRAC,
            laser_color=LASER_COLOR,
            patch_pad=PATCH_PAD,
        )

        full_x, full_y = roi_result["full_xy"]
        sigma_x, sigma_y = roi_result["sigma_xy"]
        print(f"Image:        {roi_result['image_path'].name}")
        print(f"Laser color:  {roi_result['laser_color']}")
        print(f"Full-frame x: {full_x:.2f}")
        print(f"Full-frame y: {full_y:.2f}")
        print(f"Sigma x/y:    ({sigma_x:.2f}, {sigma_y:.2f})")
        print(f"FGF conf:     {roi_result['fgf_confidence']:.3f}")
        print(f"Peak score:   {roi_result['peak_score']:.3f}")

        visualize_roi_result(roi_result)


### 5) Optional batch run over all ROI presets

Once a few fish ROIs look reasonable, use this cell to compare results across the preset list.

In [ ]:
def run_batch_from_presets(paths, roi_presets, laser_color="auto"):
    rows = []
    for path in paths:
        fish_roi = roi_presets.get(path.name)
        if fish_roi is None:
            continue
        try:
            result = run_roi_first_workflow(
                path,
                fish_roi,
                pad_frac=ROI_PAD_FRAC,
                laser_color=laser_color,
                patch_pad=PATCH_PAD,
            )
            full_x, full_y = result["full_xy"]
            rows.append({
                "file": path.name,
                "laser_color": result["laser_color"],
                "x": round(full_x, 2),
                "y": round(full_y, 2),
                "fgf_conf": round(result["fgf_confidence"], 3),
                "peak_score": round(result["peak_score"], 3),
            })
        except Exception as exc:
            rows.append({
                "file": path.name,
                "laser_color": "error",
                "x": None,
                "y": None,
                "fgf_conf": None,
                "peak_score": str(exc),
            })
    return rows


batch_rows = run_batch_from_presets(paths, ROI_PRESETS, laser_color=LASER_COLOR)

if not batch_rows:
    print("No ROI presets are available yet.")
else:
    cols = ["file", "laser_color", "x", "y", "fgf_conf", "peak_score"]
    widths = [max(len(col), max(len(str(row[col])) for row in batch_rows)) for col in cols]
    print(" | ".join(col.ljust(widths[i]) for i, col in enumerate(cols)))
    print("-+-".join("-" * widths[i] for i in range(len(cols))))
    for row in batch_rows:
        print(" | ".join(str(row[col]).ljust(widths[i]) for i, col in enumerate(cols)))
